05 - LLM labels for the whole corpus
#
# Only worth the GPU time once notebook 04 shows the model beating the
# rule-based labeller on the 58 gold studies. It did: 0.811 against 0.738,
# and 0.844 blended.
#
# Writes incrementally, so the output is usable even if the session is cut
# short -- build_weak_labels falls back to the regex score for any study the
# LLM has no row for.

In [ ]:
import sys, subprocess
from pathlib import Path

CODE = next(
    (p for pat in ("rsna-knee-code", "*/rsna-knee-code", "*/*/rsna-knee-code")
     for p in sorted(Path("/kaggle/input").glob(pat)) if (p / "src").is_dir()),
    None,
)
assert CODE is not None, "rsna-knee-code dataset is not attached to this notebook"
sys.path.insert(0, str(CODE / "src"))
sys.path.insert(0, str(CODE))
print("CODE", CODE)

# timm ships in the Kaggle image; the install is a fallback for images that lack
# it, and cannot run in the offline submission notebook.
try:
    import timm
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=False)
    import timm

import rsna_knee.config as cfg
print("RAW ", cfg.RAW, cfg.RAW.exists())
print("PREP", cfg.PREP, cfg.PREP.exists())
print("SIZE", cfg.SIZE, cfg.MOSAIC_EXT)

In [ ]:
import subprocess, sys
try:
    import bitsandbytes  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"],
                   check=False)

MODEL = "Qwen/Qwen2.5-7B-Instruct"
BATCH = 4
QUANT = "--load-4bit"

In [ ]:
!cd {CODE} && python scripts/llm_label.py --model {MODEL} {QUANT} --batch {BATCH} \
    --out /kaggle/working/llm_labels.csv

In [ ]:
import pandas as pd
df = pd.read_csv("/kaggle/working/llm_labels.csv")
print(df.shape)
print(df.drop(columns=["StudyInstanceUID"]).mean().round(3).to_string())